# 🥉 Camada Bronze — Extração e Ingestão Bruta de Ações da B3

*Primeira etapa do pipeline (arquitetura Bronze → Prata): extrai os dados do yfinance e persiste sem nenhuma transformação de negócio.*

---

## 1. Introdução

### 🎯 Objetivo
Este notebook é a **Camada Bronze** do projeto de Análise de Ações da B3. Sua única responsabilidade é **extrair os dados brutos** do `yfinance` (cotações históricas e cadastro setorial das empresas) e **persistir essas informações exatamente como vieram da fonte**, em tabelas Delta — sem limpeza, sem tradução, sem engenharia de atributos.

### 🧱 Por que separar em Bronze e Prata?
Esse projeto segue a **arquitetura medallion**, um padrão comum em Lakehouses (Databricks/Delta Lake), que organiza os dados em camadas:

| Camada | Responsabilidade |
|---|---|
| **🥉 Bronze** (este notebook) | Ingestão bruta — os dados ficam exatamente como vieram da fonte (`yfinance`). Serve como "fonte da verdade" para reprocessamento, caso algo na Prata precise ser refeito. |
| **🥈 Prata** (próximo notebook) | Lê a Bronze, aplica limpeza, padronização (tradução de setor/indústria, tipos de dados), engenharia de atributos, e gera a tabela curada + os datasets finais (CSV para Power BI e Analytics) + a análise exploratória. |

Separar as responsabilidades dessa forma deixa o pipeline mais organizado, mais fácil de depurar (cada camada pode ser re-executada isoladamente) e segue a prática padrão de engenharia de dados em Lakehouses.

### 🛠️ Tecnologias Utilizadas
| Tecnologia | Papel nesta camada |
|---|---|
| **yfinance** | Extração de cotações históricas e dados cadastrais das empresas |
| **PySpark** | Conversão para `DataFrame` distribuído |
| **Delta Lake** | Formato de persistência das tabelas Bronze |

> ▶️ **Próximo passo:** depois de executar este notebook, execute o notebook `Finance_project_Prata.ipynb`, que lê as tabelas `bronze.cotacoes_b3` e `bronze.cadastro_empresas_b3` criadas aqui.

---

## 2. Importação das Bibliotecas

In [0]:
import os
import sys
import datetime

import pandas as pd
import yfinance as yf

from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from pyspark.sql import types 
from pyspark.sql import SparkSession

## 3. Configuração do Ambiente

Configuramos o Spark, os parâmetros gerais de extração e criamos (se ainda não existir) o schema `bronze`, onde as tabelas desta camada serão salvas.

In [0]:
# Garante que o driver do Spark use o mesmo interpretador Python deste notebook
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Em um cluster Databricks a variável `spark` já existe automaticamente;
# o getOrCreate() garante que o notebook também funcione fora do Databricks (ex.: Spark local)
spark = SparkSession.builder.appName("Bronze_Acoes_B3").getOrCreate()

# Parâmetros gerais do projeto
ANOS_HISTORICO = 10  # quantidade de anos de histórico de cotações que será baixada

# Schema da camada Bronze
# 💡 Se você já usa outro nome de schema/catalog no seu workspace, basta ajustar aqui
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

DataFrame[]

## 4. Extração dos Dados

### 4.1 yfinance
O [`yfinance`](https://pypi.org/project/yfinance/) é uma biblioteca Python que consulta gratuitamente o Yahoo Finance, permitindo baixar cotações históricas (abertura, máxima, mínima, fechamento e volume) e também informações cadastrais das empresas, como setor e indústria de atuação. Para ações negociadas na B3, o ticker é sufixado com `.SA` (ex.: `PETR4.SA`).

### 4.2 Lista de Empresas da B3

As empresas foram organizadas manualmente em 6 grandes segmentos da economia, para facilitar comparações setoriais ao longo da análise: **Financeiro**, **Industrial / Materiais Básicos**, **Energia**, **Varejo**, **Tecnologia / Educação** e **Alimentos**.

> 💡 Alguns tickers aparecem comentados (`#`) intencionalmente — empresas com IPO recente, processo de recuperação judicial/delisting ou sem histórico confiável no período de 10 anos analisado.

In [0]:
finance = [
'BBDC4.SA', 'BBDC3.SA',                   #Bradesco
'ITUB4.SA', 'ITUB3.SA',                   #Itau Unibanco
'ITSA4.SA', 'ITSA3.SA',                   #Itausa (holding)
'BBAS3.SA',                               #Banco do Brasil
'SANB11.SA', 'SANB4.SA', 'SANB3.SA',      #Santander Brasil
#'BPAN4.SA',                              #Banco Pan
'ABCB4.SA',                               #Banco ABC Brasil
'BMGB4.SA',                               #Banco BMG
'BRSR6.SA', 'BRSR3.SA', 'BRSR5.SA',       #Banrisul
'BMEB4.SA', 'BMEB3.SA',                   #Banco Mercantil do Brasil
'BMIN4.SA','BMIN3.SA',                    #Banco Mercantil de Investimentos
'BAZA3.SA',                               #Banco da Amazonia
'BGIP4.SA', 'BGIP3.SA',                   #Banco Banese
'BEES3.SA', 'BEES4.SA'                    #Banco Banestes
            ]

In [0]:
industria = [
'VALE3.SA',                               #Vale (mineracao)
'USIM5.SA', 'USIM3.SA',                   #Usiminas
'CSNA3.SA',                               #CSN
'GGBR4.SA', 'GGBR3.SA',                   #Gerdau
'GOAU4.SA', 'GOAU3.SA',                   #Metalurgica Gerdau
'CMIN3.SA'                                #CSN Mineracao
              ]

In [0]:
energi = [
'PETR4.SA', 'PETR3.SA',                   #Petrobras
'PRIO3.SA',                               #PetroRio
'RAIZ4.SA',                               #Raizen
'UGPA3.SA',                               #Ultrapar
'ENEV3.SA',                               #Eneva
'EQTL3.SA',                               #Equatorial Energia
'CMIG4.SA', 'CMIG3.SA'                    #Cemig
#'ELET3.SA', 'ELET6.SA', 'ELET5.SA'        #Eletrobras
           ]

In [0]:
varejo = [
'MGLU3.SA',                               #Magazine Luiza
'LREN3.SA',                               #Lojas Renner
#'PETZ3.SA',                              #Petz
'AMER3.SA',                               #Americanas
'ASAI3.SA',                               #Assai
'PCAR3.SA',                               #Pao de Acucar
'VAMO3.SA',                               #Vamos Locadora
'RENT3.SA'                                #Localiza
           ]

In [0]:
tecnologia = [
'TOTS3.SA',                                #Totvs
'LWSA3.SA',                                #Locaweb
'YDUQ3.SA',                                #Yduqs
'COGN3.SA',                                #Cogna
'ANIM3.SA'                                 #Anima Educacao
               ]

In [0]:
food = [
'ABEV3.SA',                                #Ambev
#'BRFS3.SA',                               #BRF
#'MRFG3.SA',                               #Marfrig
'BEEF3.SA'                                 #Minerva
         ]

In [0]:
empresas = finance + industria + energi + varejo + tecnologia + food
print(f"Total de tickers mapeados: {len(empresas)}")

Total de tickers mapeados: 56


### 4.3 Download dos Dados

Com a lista de tickers definida, baixamos o histórico de preços dos últimos `ANOS_HISTORICO` anos via `yfinance` e, em seguida, coletamos as informações cadastrais (setor, indústria e nome da empresa) de cada ticker.

In [0]:
hoje = datetime.datetime.today().strftime('%Y-%m-%d')
inicio = (datetime.datetime.today() - datetime.timedelta(days=365 * ANOS_HISTORICO)).strftime('%Y-%m-%d')

dados = yf.download(empresas, start=inicio, end=hoje)
dados.head()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47489fb-6c2b-44d5-a890-a0779736817d/lib/python3.11/site-packages/yfinance/scrapers/history.py:396: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
[                       0%                       ]/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47489fb-6c2b-44d5-a890-a0779736817d/lib/python3.11/site-packages/yfinance/scrapers/history.py:396: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-f47489fb-6c2b-44d5-a890-a0779736817d/lib/python3.11/site-packages/yfinance/scrapers/history.py:396: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype ex

Price          Close                          ...   Volume                  
Ticker      ABCB4.SA   ABEV3.SA     AMER3.SA  ... VALE3.SA VAMO3.SA YDUQ3.SA
Date                                          ...                           
2016-06-30  6.034337  12.695859   881.674744  ...  6942100      NaN  4492500
2016-07-01  6.063914  12.809157   868.543396  ...  5214800      NaN  4623700
2016-07-04  6.083412  12.702520   881.674744  ...  1574800      NaN  1576000
2016-07-05  6.195527  12.855805   992.353027  ...  5508200      NaN  1756300
2016-07-06  6.117534  12.782497  1064.575317  ...  4120100      NaN  2890100

[5 rows x 280 columns]

In [0]:
dados_setores = []

for ticker in empresas:
    try:
        ticker_obj = yf.Ticker(ticker)
        info = ticker_obj.info

        dados_setores.append({
            'Codigo': ticker,
            'Setor': info.get('sector', 'Indefinido'),
            'Industria': info.get('industry', 'Indefinida'),
            'Nome_Empresa': info.get('longName', 'Desconhecido')
        })

    except Exception as e:
        print(f"Erro ao buscar {ticker}: {e}")

print(f"Informações cadastrais coletadas para {len(dados_setores)} tickers")

Informações cadastrais coletadas para 56 tickers


## 5. Estruturação Mínima e Persistência na Camada Bronze

O `yfinance` retorna os preços em formato "largo" (uma coluna por ticker, em um índice de colunas multinível). Esse formato não pode ser salvo diretamente como uma tabela. Por isso, aplicamos aqui **apenas uma transformação estrutural** — não de negócio — para "despivotar" os dados em formato longo (uma linha por combinação data + ticker) e conseguir persisti-los.

> ⚠️ Nenhuma limpeza, tradução ou padronização é feita aqui — isso é responsabilidade da Camada Prata. A Bronze preserva os dados o mais próximo possível da fonte original.

In [0]:
# "Despivota" o DataFrame: de colunas (Open, High, Low, Close, Volume) x Ticker
# para um formato longo, com uma linha por combinação (data, ticker)
dados_empilhado = dados.stack(1).reset_index()

df_dados = spark.createDataFrame(dados_empilhado)
df_setores = spark.createDataFrame(pd.DataFrame(dados_setores))

print(f"Linhas em df_dados (cotações): {df_dados.count()}")
print(f"Linhas em df_setores (cadastro): {df_setores.count()}")

Linhas em df_dados (cotações): 132902
Linhas em df_setores (cadastro): 56


In [0]:
df_dados = df_dados.select(F.col("Date").alias("data"),
                           F.col("Ticker"),
                           F.col("Open").alias("Abertura"),
                           F.col("High").alias("Maxima"),
                           F.col("Low").alias("Minima"),
                           F.col("Close").alias("Fechamento"),
                           F.col("Volume").alias("Vol")
                           )

### Persistência das Tabelas Bronze

Salvamos as duas tabelas brutas em Delta Lake, no schema `bronze`. A Camada Prata lerá essas tabelas como ponto de partida.

In [0]:
display(df_dados.limit(10))

data,Ticker,Abertura,Maxima,Minima,Fechamento,Vol
2016-06-30T00:00:00.000Z,ABCB4.SA,5.946615065268929,6.094356916057,5.909679246973035,6.034336566925049,108328.0
2016-06-30T00:00:00.000Z,ABEV3.SA,12.662536998059409,12.735846829553397,12.529246626461466,12.6958589553833,1.53283E7
2016-06-30T00:00:00.000Z,AMER3.SA,872.2952079166757,886.3644816035028,853.5361962786897,881.6747436523438,3005.0
2016-06-30T00:00:00.000Z,ANIM3.SA,3.8794462966093537,3.996079921722412,3.7322668668498062,3.996079921722412,671400.0
2016-06-30T00:00:00.000Z,BAZA3.SA,12.430445671081543,12.430445671081543,12.016097561774584,12.430445671081543,1462.0
2016-06-30T00:00:00.000Z,BBAS3.SA,4.466000132620684,4.661354064941406,4.403595634245037,4.661354064941406,2.8466E7
2016-06-30T00:00:00.000Z,BBDC3.SA,7.774569926256297,7.9614168602331015,7.7103415449285855,7.885510444641113,2919210.0
2016-06-30T00:00:00.000Z,BBDC4.SA,7.391417816624333,7.605059821775663,7.358778415238331,7.471534252166748,3.8885892E7
2016-06-30T00:00:00.000Z,BEEF3.SA,6.782286542239876,6.902860164642334,6.7069270232693405,6.902860164642334,1475549.0
2016-06-30T00:00:00.000Z,BEES3.SA,0.9994045495986938,0.9994045495986938,0.9994045495986938,0.9994045495986938,2860.0


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.table_ing_b3 (
  data TIMESTAMP,
  Ticker STRING,
  Fechamento DECIMAL(32,6),
  Maxima  DECIMAL(32,6),
  Minima   DECIMAL(32,6),
  Abertura  DECIMAL(32,6),
  Vol BIGINT
)
USING DELTA;

In [0]:
df_dados.createOrReplaceTempView("df_dados")

spark.sql("""
  INSERT OVERWRITE TABLE workspace.bronze.table_ing_b3
  SELECT * FROM df_dados
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(df_setores.limit(10))

Codigo,Setor,Industria,Nome_Empresa
BBDC4.SA,Financial Services,Banks - Regional,Banco Bradesco S.A.
BBDC3.SA,Financial Services,Banks - Regional,Banco Bradesco S.A.
ITUB4.SA,Financial Services,Banks - Regional,Itaú Unibanco Holding S.A.
ITUB3.SA,Financial Services,Banks - Regional,Itaú Unibanco Holding S.A.
ITSA4.SA,Industrials,Conglomerates,Itaúsa S.A.
ITSA3.SA,Industrials,Conglomerates,Itaúsa S.A.
BBAS3.SA,Financial Services,Banks - Regional,Banco do Brasil S.A.
SANB11.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.
SANB4.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.
SANB3.SA,Financial Services,Banks - Regional,Banco Santander (Brasil) S.A.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze.table_cadastro_empresas (
  Codigo STRING,
  Setor STRING,
  Industria STRING,
  Nome_Empresa STRING
)
USING DELTA;

In [0]:
df_setores.createOrReplaceTempView("df_setores")

spark.sql("""
  INSERT OVERWRITE TABLE workspace.bronze.table_cadastro_empresas
  SELECT * FROM df_setores
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
#df_dados.write.mode("overwrite").saveAsTable("workspace.bronze.table_ing_b3")
#df_setores.write.mode("overwrite").saveAsTable("workspace.broze.table_cadastro_empresas")

print("Tabelas Bronze salvas com sucesso:")
print(" - bronze.cotacoes_b3")
print(" - bronze.cadastro_empresas_b3")

Tabelas Bronze salvas com sucesso:
 - bronze.cotacoes_b3
 - bronze.cadastro_empresas_b3


## Conclusão da Camada Bronze

Este notebook extraiu, via `yfinance`, o histórico de cotações e o cadastro setorial de mais de 50 ações da B3, e persistiu essas informações sem nenhum tratamento de negócio em duas tabelas Delta:

- **`bronze.cotacoes_b3`** — cotações diárias (abertura, máxima, mínima, fechamento, volume) por ticker;
- **`bronze.cadastro_empresas_b3`** — setor, indústria e nome de cada empresa, em inglês (como retornado pelo `yfinance`).

➡️ **Próximo notebook:** `Finance_project_Prata.ipynb`, que lê essas duas tabelas, aplica limpeza, padronização e engenharia de atributos, e gera a tabela curada e os datasets finais.